In [1]:
!pip -q install -U openai langchain langchain-openai langchain-community faiss-cpu pandas

In [2]:
import os
import getpass
import pandas as pd

from openai import OpenAI

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

/tmp/ipykernel_2638/1647089152.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
# Enter your OpenAI API key when prompted
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
documents = [
    Document(
        page_content="""
        NovaTech Solutions is a fictional AI software company.
        Its next-generation AI platform has the internal codename Project Orion.
        Project Orion is planned as a multimodal enterprise assistant.
        """,
        metadata={"source": "company_overview.txt"}
    ),

    Document(
        page_content="""
        NovaTech Solutions pricing policy states that annual plans receive
        a 20% discount compared with the equivalent monthly plan.
        The discount applies to standard annual subscriptions.
        """,
        metadata={"source": "pricing_policy.txt"}
    ),

    Document(
        page_content="""
        NovaTech Solutions operates its fictional R&D center in Pune, India.
        The R&D center focuses on natural language processing, retrieval systems,
        and enterprise AI.
        """,
        metadata={"source": "locations.txt"}
    ),

    Document(
        page_content="""
        For critical security incidents, NovaTech Solutions uses the escalation
        email security-escalation@novatech.com.
        This address is reserved for high-priority security incidents.
        """,
        metadata={"source": "security_guidelines.txt"}
    ),

    Document(
        page_content="""
        Under NovaTech Solutions' fictional free plan, each user may create
        a maximum of 5 support tickets per month.
        Additional support tickets require an upgraded plan.
        """,
        metadata={"source": "support_policy.txt"}
    ),

    Document(
        page_content="""
        NovaTech Solutions retains normal application logs for 30 days.
        Normal system logs are automatically deleted after the retention period.
        """,
        metadata={"source": "logging_policy.txt"}
    ),

    Document(
        page_content="""
        NovaTech Solutions retains audit logs for 180 days.
        Audit logs have a longer retention period than normal application logs
        because they are used for compliance investigations.
        """,
        metadata={"source": "audit_policy.txt"}
    ),

    Document(
        page_content="""
        NovaTech Solutions is NOT SOC 2 certified.
        The company is currently preparing internal controls for a future
        SOC 2 assessment.
        """,
        metadata={"source": "compliance_status.txt"}
    ),

    Document(
        page_content="""
        NovaTech Solutions' fictional headquarters is located in Bengaluru.
        The R&D center, however, is located separately in Pune.
        """,
        metadata={"source": "office_locations.txt"}
    ),
]

print(f"Knowledge-base documents: {len(documents)}")

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = FAISS.from_documents(
    documents,
    embeddings
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("FAISS vector store created successfully.")

In [ ]:
def retrieve_context(query, k=3):
    results = vector_store.similarity_search(query, k=k)

    return results

In [ ]:
query = "What is NovaTech's internal codename for its next-generation AI platform?"

results = retrieve_context(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content.strip())
    print("Source:", doc.metadata["source"])

In [ ]:
RAG_PROMPT = """
You are a factual question-answering assistant.

Use ONLY the information contained in the CONTEXT to answer the QUESTION.

Rules:
1. Do not use outside knowledge.
2. Do not guess.
3. If the answer is not supported by the context, say:
   "I don't have enough information in the provided context."
4. Keep the answer concise.
5. Pay close attention to words such as NOT, NEVER, ONLY, and NO.

CONTEXT:
----------------
{context}
----------------

QUESTION:
{question}

ANSWER:
"""

In [ ]:
def generate_with_rag(query):
    retrieved_docs = retrieve_context(query, k=3)

    context = "\n\n".join(
        [
            f"[Source: {doc.metadata.get('source', 'unknown')}]\n"
            f"{doc.page_content.strip()}"
            for doc in retrieved_docs
        ]
    )

    prompt = RAG_PROMPT.format(
        context=context,
        question=query
    )

    response = client.chat.completions.create(
        model="gpt-5.6",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": "Answer strictly from the supplied context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
def generate_without_rag(query):
    response = client.chat.completions.create(
        model="gpt-5.6",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the user's question. "
                    "Do not claim certainty about fictional company facts."
                )
            },
            {
                "role": "user",
                "content": query
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
test_query = (
    "What is NovaTech Solutions' internal codename "
    "for its next-generation AI platform?"
)

print("===== WITHOUT RAG =====")
print(generate_without_rag(test_query))

print("\n===== WITH RAG =====")
print(generate_with_rag(test_query))

In [ ]:
test_queries = [
    "What is NovaTech Solutions' internal codename for its next-generation AI platform?",
    "According to the internal pricing policy, what discount is given for annual plans?",
    "Where is NovaTech Solutions' R&D center located?",
    "What is the escalation email for critical security incidents?",
    "What is the maximum number of support tickets allowed per user per month under the free plan?"
]

In [ ]:
results = []

for i, query in enumerate(test_queries, 1):

    without_rag = generate_without_rag(query)
    with_rag = generate_with_rag(query)

    results.append({
        "Query": query,
        "Without RAG": without_rag,
        "With RAG": with_rag
    })

comparison_df = pd.DataFrame(results)

pd.set_option("display.max_colwidth", 500)

comparison_df

In [ ]:
evaluation = [
    {
        "Query": test_queries[0],
        "Ground Truth": "Project Orion",
        "RAG Grounded?": "Yes",
        "No-RAG Grounded?": "Usually No"
    },
    {
        "Query": test_queries[1],
        "Ground Truth": "20%",
        "RAG Grounded?": "Yes",
        "No-RAG Grounded?": "Usually No"
    },
    {
        "Query": test_queries[2],
        "Ground Truth": "Pune, India",
        "RAG Grounded?": "Yes",
        "No-RAG Grounded?": "Usually No"
    },
    {
        "Query": test_queries[3],
        "Ground Truth": "security-escalation@novatech.com",
        "RAG Grounded?": "Yes",
        "No-RAG Grounded?": "Usually No"
    },
    {
        "Query": test_queries[4],
        "Ground Truth": "5 tickets/month",
        "RAG Grounded?": "Yes",
        "No-RAG Grounded?": "Usually No"
    }
]

evaluation_df = pd.DataFrame(evaluation)
evaluation_df

In [ ]:
failure_query_1 = (
    "What is the data retention period for audit logs?"
)

retrieved = retrieve_context(failure_query_1, k=1)

print("QUERY:")
print(failure_query_1)

print("\nRETRIEVED CHUNK:")
print(retrieved[0].page_content)

print("\nRAG ANSWER:")
print(generate_with_rag(failure_query_1))

In [ ]:
failure_query_2 = (
    "Is NovaTech Solutions SOC 2 certified?"
)

retrieved = retrieve_context(failure_query_2, k=3)

print("QUERY:")
print(failure_query_2)

print("\nRETRIEVED CHUNKS:")

for i, doc in enumerate(retrieved, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content)